# Doc-RAG: Complete Backend Source Code Backup
This notebook contains the complete consolidated source code of the Doc-RAG backend application, organized into modules for reference, prototyping, and research.

## Module 1: Core Configuration & Security
Handles environment variables, settings, and JWT authentication logic.

In [ ]:
# app/core/config.py and security.py logic
import os
from pydantic_settings import BaseSettings
from datetime import datetime, timedelta
from jose import JWTError, jwt
from passlib.context import CryptContext

class Settings(BaseSettings):
    PINECONE_INDEX_NAME: str = "doc-rag"
    JWT_SECRET: str = "88d6b113-b342-4358-9c87-82344f77f7dd"
    ALGORITHM: str = "HS256"
    
settings = Settings()
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def verify_password(plain_password, hashed_password):
    return pwd_context.verify(plain_password, hashed_password)

def create_access_token(data: dict):
    to_encode = data.copy()
    expire = datetime.utcnow() + timedelta(days=30)
    to_encode.update({"exp": expire})
    return jwt.encode(to_encode, settings.JWT_SECRET, algorithm=settings.ALGORITHM)

## Module 2: Vector Store (Pinecone & BM25)
Logic for hybrid search, embedding generation, and vector indexing.

In [ ]:
# app/services/vector_store.py
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(settings.PINECONE_INDEX_NAME)
bm25 = BM25Encoder().default()

def query_hybrid(dense_vector, sparse_vector, top_k=5, namespace="default"):
    return index.query(
        vector=dense_vector,
        sparse_vector=sparse_vector,
        top_k=top_k,
        include_metadata=True,
        namespace=namespace
    )

## Module 3: LLM & Agentic Services
Multi-provider support (Gemini/Groq/OpenAI/Ollama) and the Weather/Web-Search agent.

In [ ]:
# app/services/llm.py and agent.py
import google.genai as genai
from google.genai import types as genai_types

WEATHER_SYSTEM_PROMPT = "You are a professional Weather Assistant..."

async def run_weather_agent(question, history=None):
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"), http_options={"api_version": "v1beta"})
    # Tool definitions and stream logic here...
    pass

## Module 4: API Business Logic
Core logic for document uploads, PDF processing, and session history management.

In [ ]:
# app/api/v1/documents.py and sessions.py
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_text(text):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    return splitter.split_text(text)

def extract_pdf_text(stream):
    reader = PdfReader(stream)
    return " ".join([page.extract_text() for page in reader.pages])